# B1 — Tune extraction prompt against Apollodorus

Per `docs/TODO-stage4.md` Track B: tune the extraction prompt against Apollodorus (the
spine source, most systematic) on 5-10 segments before running the full corpus (B2).
This is the highest-value 30 minutes in Stage 4 — if extraction quality is good here,
the rest follows.

**Before running:** make sure `.env` (repo root) has `ANTHROPIC_API_KEY` and
`EXTRACTION_MODEL` uncommented and filled in — this notebook makes real API calls that
cost real money (Opus 4.8, the strongest tier per ADR-008).

**Workflow:** run the cells below, eyeball the extracted facts against the segment
text, and if quality is off, edit `SOURCE_HINTS`/`SYSTEM_PROMPT`/`max_tokens` in
`extraction/claim_extractor.py`, then run the **reload cell** below (not a full kernel
restart -- that would throw away the fetch cell's resumed progress in `results`) and
re-run the fetch cell. Once Apollodorus looks good, move on to B2:
`cd ingestion && python -m extraction.run_extraction`.

In [1]:
import sys
from pathlib import Path

INGESTION_ROOT = Path.cwd().parent  # ingestion/ -- this notebook lives in ingestion/notebooks/
sys.path.insert(0, str(INGESTION_ROOT))

from dotenv import load_dotenv
load_dotenv()  # walks up from cwd and finds the repo-root .env

True

In [7]:
from extraction.claim_extractor import extract_facts
from extraction.segmentation import segment
from loader.source_registry import SOURCE_REGISTRY
from loader.text_cleaner import clean

apollodorus = next(s for s in SOURCE_REGISTRY if s.source_id == "apollodorus-bibliotheca")
apollodorus

SourceConfig(source_id='apollodorus-bibliotheca', author='Apollodorus', work='Bibliotheca', file_path='corpus/apollodorus_bibliotheca_frazer1921.txt', passage_ref_extractor=<function apollodorus_refs at 0x10d65e2a0>)

In [3]:
raw = (INGESTION_ROOT / apollodorus.file_path).read_text(encoding="utf-8")
cleaned = clean(raw)
segments = segment(cleaned, apollodorus.author, apollodorus.work, apollodorus.passage_ref_extractor)
print(f"{len(segments)} segments total")

115 segments total


In [4]:
# Edit this to try a different slice/sample once you've looked at the first batch.
SAMPLE = segments[:8]
[s.passage_ref for s in SAMPLE]

['Apollodorus, Bibliotheca',
 '1.1.1-1.1.7',
 '1.2.1-1.2.7',
 '1.3.1-1.3.5',
 '1.3.6-1.4.2',
 '1.4.3-1.5.2',
 '1.5.3-1.6.2',
 '1.6.3']

In [8]:
# Run this after editing extraction/claim_extractor.py (SOURCE_HINTS, SYSTEM_PROMPT,
# max_tokens, ...) and before re-running the fetch cell below. `from ... import
# extract_facts` in an earlier cell bound the OLD function object -- editing the .py
# file on disk doesn't change what's already in this kernel's memory. importlib.reload
# re-executes the module (rebinding extract_facts to the edited version) without
# restarting the kernel, so the `results` dict's already-completed segments survive.
import importlib

import extraction.claim_extractor
importlib.reload(extraction.claim_extractor)
from extraction.claim_extractor import extract_facts

In [10]:
# `results` persists across re-runs of this cell within the current kernel
# session (only initialized once, via the try/except below) -- if extract_facts
# raises partway through (e.g. IncompleteOutputException on one dense segment),
# already-processed segments stay cached here. Re-running this cell picks up
# only where it left off instead of re-paying for every prior segment.
#
# To force a full re-run from scratch, run `results = {}` in a cell first.
try:
    results
except NameError:
    results = {}

for i, seg in enumerate(SAMPLE):
    if i in results:
        continue
    print(f"extracting [{i}] passage_ref={seg.passage_ref} ...")
    results[i] = extract_facts(seg.text, apollodorus.source_id)

print(f"{len(results)}/{len(SAMPLE)} segments done")

extracting [2] passage_ref=1.2.1-1.2.7 ...
extracting [3] passage_ref=1.3.1-1.3.5 ...
extracting [4] passage_ref=1.3.6-1.4.2 ...
extracting [5] passage_ref=1.4.3-1.5.2 ...
extracting [6] passage_ref=1.5.3-1.6.2 ...
extracting [7] passage_ref=1.6.3 ...
8/8 segments done


In [14]:
print("opus")
# Pure display, no API calls -- safe to re-run any time regardless of what
# the fetch cell above has completed so far.
for i, seg in enumerate(SAMPLE):
    print("=" * 80)
    print(f"[{i}] passage_ref: {seg.passage_ref}")
    print(f"segment text ({len(seg.text)} chars):")
    print(seg.text)
    # print(seg.text[:500] + ("..." if len(seg.text) > 500 else ""))
    print()

    if i not in results:
        print("(not extracted yet -- re-run the cell above)")
        print()
        continue

    facts = results[i]
    print(f"entities ({len(facts.entities)}):")
    for e in facts.entities:
        print(f"  - {e.name} ({e.type}, generation={e.generation}, domain={e.domain})")

    print(f"relationships ({len(facts.relationships)}):")
    for r in facts.relationships:
        flag = " [contested]" if r.is_contested else ""
        print(f"  - {r.from_name} --{r.relation}--> {r.to_name}{flag}")

    print(f"variant_claims ({len(facts.variant_claims)}):")
    for c in facts.variant_claims:
        print(f"  - [{c.claim_type}] {c.subject_name}: {c.claim_value}")
    print()

opus
[0] passage_ref: Apollodorus, Bibliotheca
segment text (154 chars):
Translated by Sir James George Frazer (1921)
Source: https://www.theoi.com/Text/Apollodorus1.html
Public domain (Loeb Classical Library, 1921 translation)

entities (0):
relationships (0):
variant_claims (0):

[1] passage_ref: 1.1.1-1.1.7
segment text (2085 chars):
Sky was the first who ruled over the whole world. And having wedded Earth, he begat first the Hundred-handed, as they are named: Briareus, Gyes, Cottus, who were unsurpassed in size and might, each of them having a hundred hands and fifty heads.
After these, Earth bore him the Cyclopes, to wit, Arges, Steropes, Brontes of whom each had one eye on his forehead. But them Sky bound and cast into Tartarus, a gloomy place in Hades as far distant from earth as earth is distant from the sky.
And again he begat children by Earth, to wit, the Titans as they are named: Ocean, Coeus, Hyperion, Crius, Iapetus, and, youngest of all, Cronus; also daughters, the Titan

## Compare against another model

Same segments (`SAMPLE`), same prompt -- just a different `model=`. Reuses
`extraction.claim_extractor.client` directly rather than `extract_facts()`
(which hardcodes `EXTRACTION_MODEL`), so nothing in `claim_extractor.py` needs
to change. This only works as-is for another **Anthropic** model, since
`client` is an Anthropic-backed `instructor` client
(`instructor.from_anthropic(...)`); to compare a different provider, build a
second client the same way instructor supports it for that provider (e.g.
`instructor.from_openai(OpenAI(api_key=os.environ["OPENAI_API_KEY"]))`) --
the `ExtractedFacts` response model and the prompt are portable across
providers unchanged, only the client construction and `model=` differ.

In [12]:
# Edit this to whatever you want to compare against.
OTHER_MODEL = "claude-sonnet-5"

# Same resumable pattern as the fetch cell above -- safe to re-run after a
# partial failure without re-paying for already-extracted segments.
try:
    results_other_model
except NameError:
    results_other_model = {}

for i, seg in enumerate(SAMPLE):
    if i in results_other_model:
        continue
    hint = extraction.claim_extractor.SOURCE_HINTS.get(apollodorus.source_id)
    system = (
        f"{extraction.claim_extractor.SYSTEM_PROMPT}\n\n{hint}"
        if hint
        else extraction.claim_extractor.SYSTEM_PROMPT
    )
    print(f"extracting [{i}] passage_ref={seg.passage_ref} with {OTHER_MODEL} ...")
    results_other_model[i] = extraction.claim_extractor.client.chat.completions.create(
        model=OTHER_MODEL,
        response_model=extraction.claim_extractor.ExtractedFacts,
        max_tokens=16000,
        system=system,
        messages=[{"role": "user", "content": seg.text}],
    )

print(f"{len(results_other_model)}/{len(SAMPLE)} segments done with {OTHER_MODEL}")

extracting [0] passage_ref=Apollodorus, Bibliotheca with claude-sonnet-5 ...
extracting [1] passage_ref=1.1.1-1.1.7 with claude-sonnet-5 ...
extracting [2] passage_ref=1.2.1-1.2.7 with claude-sonnet-5 ...
extracting [3] passage_ref=1.3.1-1.3.5 with claude-sonnet-5 ...
extracting [4] passage_ref=1.3.6-1.4.2 with claude-sonnet-5 ...
extracting [5] passage_ref=1.4.3-1.5.2 with claude-sonnet-5 ...
extracting [6] passage_ref=1.5.3-1.6.2 with claude-sonnet-5 ...
extracting [7] passage_ref=1.6.3 with claude-sonnet-5 ...
8/8 segments done with claude-sonnet-5


In [15]:
print(OTHER_MODEL)
# Pure display, no API calls -- safe to re-run any time regardless of what
# the fetch cell above has completed so far.
for i, seg in enumerate(SAMPLE):
    print("=" * 80)
    print(f"[{i}] passage_ref: {seg.passage_ref}")
    print(f"segment text ({len(seg.text)} chars):")
    print(seg.text)
    # print(seg.text[:500] + ("..." if len(seg.text) > 500 else ""))
    print()

    if i not in results_other_model:
        print("(not extracted yet -- re-run the cell above)")
        print()
        continue

    facts = results_other_model[i]
    print(f"entities ({len(facts.entities)}):")
    for e in facts.entities:
        print(f"  - {e.name} ({e.type}, generation={e.generation}, domain={e.domain})")

    print(f"relationships ({len(facts.relationships)}):")
    for r in facts.relationships:
        flag = " [contested]" if r.is_contested else ""
        print(f"  - {r.from_name} --{r.relation}--> {r.to_name}{flag}")

    print(f"variant_claims ({len(facts.variant_claims)}):")
    for c in facts.variant_claims:
        print(f"  - [{c.claim_type}] {c.subject_name}: {c.claim_value}")
    print()

claude-sonnet-5
[0] passage_ref: Apollodorus, Bibliotheca
segment text (154 chars):
Translated by Sir James George Frazer (1921)
Source: https://www.theoi.com/Text/Apollodorus1.html
Public domain (Loeb Classical Library, 1921 translation)

entities (0):
relationships (0):
variant_claims (0):

[1] passage_ref: 1.1.1-1.1.7
segment text (2085 chars):
Sky was the first who ruled over the whole world. And having wedded Earth, he begat first the Hundred-handed, as they are named: Briareus, Gyes, Cottus, who were unsurpassed in size and might, each of them having a hundred hands and fifty heads.
After these, Earth bore him the Cyclopes, to wit, Arges, Steropes, Brontes of whom each had one eye on his forehead. But them Sky bound and cast into Tartarus, a gloomy place in Hades as far distant from earth as earth is distant from the sky.
And again he begat children by Earth, to wit, the Titans as they are named: Ocean, Coeus, Hyperion, Crius, Iapetus, and, youngest of all, Cronus; also daughters

In [13]:
# Pure diff, no API calls -- compares the baseline `results` (EXTRACTION_MODEL)
# against `results_other_model` (OTHER_MODEL) per segment.
for i, seg in enumerate(SAMPLE):
    print("=" * 80)
    print(f"[{i}] passage_ref: {seg.passage_ref}")

    if i not in results or i not in results_other_model:
        print("(missing from one side -- run both fetch cells first)")
        continue

    base = results[i]
    other = results_other_model[i]

    base_names = {e.name for e in base.entities}
    other_names = {e.name for e in other.entities}
    print(f"entities: {len(base_names)} (baseline) vs {len(other_names)} ({OTHER_MODEL})")
    if base_names != other_names:
        print(f"  only in baseline: {sorted(base_names - other_names)}")
        print(f"  only in {OTHER_MODEL}: {sorted(other_names - base_names)}")

    base_rels = {(r.from_name, r.relation, r.to_name) for r in base.relationships}
    other_rels = {(r.from_name, r.relation, r.to_name) for r in other.relationships}
    print(f"relationships: {len(base_rels)} (baseline) vs {len(other_rels)} ({OTHER_MODEL})")
    if base_rels != other_rels:
        print(f"  only in baseline: {sorted(base_rels - other_rels)}")
        print(f"  only in {OTHER_MODEL}: {sorted(other_rels - base_rels)}")

    base_claims = {(c.claim_type, c.subject_name, c.claim_value) for c in base.variant_claims}
    other_claims = {(c.claim_type, c.subject_name, c.claim_value) for c in other.variant_claims}
    print(f"variant_claims: {len(base_claims)} (baseline) vs {len(other_claims)} ({OTHER_MODEL})")
    if base_claims != other_claims:
        print(f"  only in baseline: {sorted(base_claims - other_claims)}")
        print(f"  only in {OTHER_MODEL}: {sorted(other_claims - base_claims)}")
    print()

[0] passage_ref: Apollodorus, Bibliotheca
entities: 0 (baseline) vs 0 (claude-sonnet-5)
relationships: 0 (baseline) vs 0 (claude-sonnet-5)
variant_claims: 0 (baseline) vs 0 (claude-sonnet-5)

[1] passage_ref: 1.1.1-1.1.7
entities: 39 (baseline) vs 35 (claude-sonnet-5)
  only in baseline: ['Crete', 'Dicte', 'Hades', 'Tartarus']
  only in claude-sonnet-5: []
relationships: 57 (baseline) vs 53 (claude-sonnet-5)
  only in baseline: [('Melisseus', 'parent_of', 'Adrastia'), ('Melisseus', 'parent_of', 'Ida'), ('Sky', 'parent_of', 'Alecto'), ('Sky', 'parent_of', 'Megaera'), ('Sky', 'parent_of', 'Tisiphone')]
  only in claude-sonnet-5: [('Sky', 'killed_by', 'Cronus')]
variant_claims: 4 (baseline) vs 5 (claude-sonnet-5)
  only in baseline: [('parentage', 'Furies', "born from the drops of blood from Sky's severed genitals"), ('parentage', 'Zeus', 'son of Cronus and Rhea, born in a cave of Dicte in Crete')]
  only in claude-sonnet-5: [('birth', 'Zeus', 'born to Rhea in a cave of Dicte in Crete, sa

## What to look for

- **Entities**: correct `type` (must match the `entities.type` CHECK values —
  `primordial`/`titan`/`olympian`/`other_god`/`hero`/`mortal`/`monster`/`nymph`), no
  obviously duplicated names for the same being.
- **Relationships**: right direction (`from_name` --relation--> `to_name`), and
  `is_contested=true` only where the passage actually signals disagreement
  ("some say", "others hold").
- **Variant claims**: every attributed claim extracted, not only explicitly-flagged
  ones (ADR-007 §1 — `is_contested` is a soft hint, never a storage gate).

Once this looks good, proceed to B2: `cd ingestion && python -m extraction.run_extraction`.